In [ ]:
import os
os.environ['EINV_CONFIG'] = 'expansion/setup05.yaml'
os.environ['TRAIN_TAG'] = 'exp05'
from utils.core.path import get_metrics_path, get_gpt_scores_path
import argparse
import random
from utils.project_utils import set_global_seed
from utils.core.files import read_json

In [ ]:
set_global_seed(42)

file = get_metrics_path(split="val", model_type='einv')
data = read_json(file)

In [ ]:
def get_gpt_scores(gpt_scores):

    file = get_gpt_scores_path(split="val", model_type='einv')
    gpt_scores = read_json(file)

    _structure_scores = []
    _entity_scores = []
    _topic_scores = []

    for score in gpt_scores:
        if 'structure_score' in score.keys() and score['structure_score'] is not None and 0 <= score['structure_score'] <= 5:
            _structure_scores.append(score['structure_score'])
        if 'entity_score' in score.keys() and score['entity_score'] is not None and 0 <= score['entity_score'] <= 5:
            _entity_scores.append(score['entity_score'])
        if 'topic_score' in score.keys() and score['topic_score'] is not None and 0 <= score['topic_score'] <=5:
            _topic_scores.append(score['topic_score'])

    gpt_scores = {
        'structure': _structure_scores,
        'entity': _entity_scores,
        'topic': _topic_scores
    }
    
    return gpt_scores

In [ ]:
gpt_scores = get_gpt_scores(gpt_scores)
metrics = {}
_metrics = {
    'loss': 'loss', 
    'perlexity': 'perplexity', 
    'rouge1': 'ROUGE-1', 
    'rouge2': 'ROUGE-2', 
    'rougeL': 'ROUGE-L', 
    'f1': 'BERTScore F1', 
    'structure': 'Structure', 
    'entity': 'Entity', 
    'topic': 'Topic'
}

for metric in _metrics.keys():
    if metric in data.keys():
        if isinstance(data[metric], list) and len(data[metric]) > 2000:
            metrics[metric] = random.sample(data[metric], 2000)
    elif metric in gpt_scores.keys():
        metrics[metric] = gpt_scores[metric]

In [ ]:
for metric in metrics.keys():
    print(f"{metric}: {len(metrics[metric])}")